In [12]:
!pip install --upgrade gdown -q

import os

if "Subtes" not in os.listdir():
    !gdown --folder https://drive.google.com/drive/folders/1N_upx66kxNa6MqUD40SsvJqioQLJ8jdm
else:
    print("La carpeta 'Subtes' ya está descargada.")


import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests

La carpeta 'Subtes' ya está descargada.


In [2]:
print("Archivos y carpetas en Colab:")
print(os.listdir('.'))

Archivos y carpetas en Colab:
['.config', 'Subtes', 'sample_data']


In [3]:
path = "Subtes/"

In [4]:
# Cargar los datasets que definimos previamente
df_historico_precio = pd.read_csv(path + 'registro-historico-del-precio-del-boleto.csv')
df_viajes_anual = pd.read_csv(path + 'viajes_anual.csv')

# Verificar que se hayan cargado correctamente
print("Precios cargados:", df_historico_precio.shape)
print("Viajes anuales cargados:", df_viajes_anual.shape)

Precios cargados: (304, 4)
Viajes anuales cargados: (48, 3)


In [5]:
# Inspección básica para ver la estructura con tus variables
print("--- Histórico Precio del Boleto ---")
display(df_historico_precio.head())
print(df_historico_precio.info())

print("\n--- Viajes Anuales ---")
display(df_viajes_anual.head())
print(df_viajes_anual.info())

--- Histórico Precio del Boleto ---


,año,mes_numero,mes,precio
0,1994,1,ENERO,0.45
1,1994,2,FEBRERO,0.45
2,1994,3,MARZO,0.45
3,1994,4,ABRIL,0.45
4,1994,5,MAYO,0.45


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   año         304 non-null    int64  
 1   mes_numero  304 non-null    int64  
 2   mes         304 non-null    object 
 3   precio      304 non-null    float64
dtypes: float64(1), int64(2), object(1)
memory usage: 9.6+ KB
None

--- Viajes Anuales ---


,year,LINEA,total
0,2013,A,26731478
1,2013,B,49458167
2,2013,C,27839180
3,2013,D,45532347
4,2013,E,11815541


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   year    48 non-null     int64 
 1   LINEA   48 non-null     object
 2   total   48 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.3+ KB
None


In [6]:
# Para el dataset de precios: crear una columna de fecha para series de tiempo
df_historico_precio['fecha'] = pd.to_datetime(df_historico_precio['año'].astype(str) + '-' + df_historico_precio['mes_numero'].astype(str) + '-01')

# Calcular el precio promedio anual del boleto para cruzar con los viajes anuales
precio_anual = df_historico_precio.groupby('año')['precio'].mean().reset_index()

In [7]:
precio_anual

,año,precio
0,1994,0.450000
1,1995,0.450000
2,1996,0.462500
3,1997,0.500000
4,1998,0.500000
5,1999,0.566667
6,2000,0.600000
7,2001,0.683333
8,2002,0.700000
9,2003,0.700000


In [8]:
fig_precio = px.line(
    df_historico_precio,
    x='fecha',
    y='precio',
    title='Evolución Histórica del Precio del Boleto',
    labels={'precio': 'Precio ($)', 'fecha': 'Fecha'},
    markers=True
)
fig_precio.update_layout(template='plotly_white')
fig_precio.show()

In [9]:
df_viajes_anual.columns

Index(['year', 'LINEA', 'total'], dtype='object')

In [10]:
df_viajes_anual = df_viajes_anual.rename(columns={'year': 'año'})

fig_viajes = px.line(
    df_viajes_anual,
    x='año',
    y='total',
    color='LINEA',
    title='Cantidad Anual de Viajes por Línea',
    labels={'total': 'Total de Viajes', 'año': 'Año'},
    markers=True
)
# Ajustar el eje X para que muestre solo años enteros
fig_viajes.update_xaxes(dtick=1)
fig_viajes.update_layout(template='plotly_white')
fig_viajes.show()

In [11]:
# Sumar todos los viajes de todas las líneas por año
viajes_totales_anuales = df_viajes_anual.groupby('año')['total'].sum().reset_index()

# Unir con el precio anual promedio que calculamos en el Paso 2
df_cruzado = pd.merge(viajes_totales_anuales, precio_anual, on='año', how='inner')

# Crear el gráfico de doble eje Y
fig_cruce = go.Figure()

# Agregar barra para los viajes totales
fig_cruce.add_trace(
    go.Bar(x=df_cruzado['año'], y=df_cruzado['total'], name='Total de Viajes', opacity=0.6)
)

# Agregar línea para el precio promedio
fig_cruce.add_trace(
    go.Scatter(x=df_cruzado['año'], y=df_cruzado['precio'], name='Precio Promedio', yaxis='y2', mode='lines+markers', line=dict(color='red', width=3))
)

# Configurar el diseño
fig_cruce.update_layout(
    title='Relación entre Viajes Totales y Precio Promedio Anual del Boleto',
    xaxis=dict(title='Año', dtick=1),
    yaxis=dict(title='Total de Viajes', side='left'),
    yaxis2=dict(title='Precio Promedio ($)', side='right', overlaying='y', showgrid=False),
    template='plotly_white',
    legend=dict(x=0.01, y=0.99) # Posición de la leyenda
)

fig_cruce.show()

In [13]:
# 1. Homogeneizar columnas
# Aseguramos que la columna de tiempo se llame 'año' en ambos
if 'year' in df_viajes_anual.columns:
    df_viajes_anual = df_viajes_anual.rename(columns={'year': 'año'})

# 2. Agrupación y unificación
# Sumamos el total de viajes de todas las líneas por año
viajes_totales = df_viajes_anual.groupby('año')['total'].sum().reset_index()

# Calculamos el precio promedio del boleto por año
precio_promedio = df_historico_precio.groupby('año')['precio'].mean().reset_index()

# Unimos ambos DataFrames
df_granger = pd.merge(viajes_totales, precio_promedio, on='año', how='inner')

# 3. Preparación de la Serie Temporal
# Ordenamos cronológicamente y seteamos el año como índice del DataFrame
df_granger = df_granger.sort_values('año').set_index('año')

# 4. Estacionariedad
# Aplicamos diferenciación (valor actual menos valor anterior) para eliminar tendencias
df_estacionario = df_granger[['total', 'precio']].diff().dropna()

# 5. Ejecución del Test de Granger
# El formato requiere que la columna 1 sea Y (lo que queremos predecir: total de viajes)
# y la columna 2 sea X (la variable causal: precio)
datos_test = df_estacionario[['total', 'precio']]

print("--- Muestra de la Serie Temporal Unificada ---")
display(df_granger.head())

print("\n--- Resultados del Test de Causalidad de Granger ---")
print("H0: El 'precio' NO causa a 'total' de viajes\n")

# NOTA: Usamos maxlag=1 o 2 (rezagos de 1 o 2 años).
# Al ser datos anuales, usar un número mayor romperá el test por falta de observaciones.
try:
    resultados = grangercausalitytests(datos_test, maxlag=2, verbose=True)
except ValueError as e:
    print(f"\n⚠️ Error de modelado: {e}")
    print("Esto suele ocurrir si hay muy pocos años en el dataset para calcular múltiples rezagos.")
    print("Intentando nuevamente solo con maxlag=1...\n")
    resultados = grangercausalitytests(datos_test, maxlag=1, verbose=True)

--- Muestra de la Serie Temporal Unificada ---


,total,precio
año,,
2013,165754193,2.666667
2014,255189625,4.333333
2015,282519120,4.500000
2016,314418191,5.000000
2017,328701725,7.500000



--- Resultados del Test de Causalidad de Granger ---
H0: El 'precio' NO causa a 'total' de viajes


⚠️ Error de modelado: Insufficient observations. Maximum allowable lag is 0
Esto suele ocurrir si hay muy pocos años en el dataset para calcular múltiples rezagos.
Intentando nuevamente solo con maxlag=1...


Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.7566  , p=0.4761  , df_denom=2, df_num=1
ssr based chi2 test:   chi2=1.8915  , p=0.1690  , df=1
likelihood ratio test: chi2=1.6042  , p=0.2053  , df=1
parameter F test:         F=0.7566  , p=0.4761  , df_denom=2, df_num=1


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/stattools.py:1556: FutureWarning:

verbose is deprecated since functions should not print results



In [17]:
import duckdb
import pandas as pd

print("Procesando los archivos masivos ignorando problemas de fechas en CSV...")

# 1. Le decimos a DuckDB que lea todo como VARCHAR (texto) para evitar que se rompa con fechas raras.
# 2. Agrupamos por año y mes como texto.
query_mensual = """
    SELECT
        COALESCE(fecha, FECHA) AS fecha_texto,
        SUM(CAST(COALESCE(pax_total, PAX_TOTAL) AS INT)) AS total_viajes
    FROM read_csv_auto('Subtes/historico_*.csv', union_by_name=True, all_varchar=True)
    GROUP BY 1
"""

# Ejecutamos la consulta. Ahora devolverá un DataFrame con 'fecha_texto' y 'total_viajes'
df_temp = duckdb.query(query_mensual).df()

# Filtramos filas vacías
df_temp = df_temp.dropna(subset=['fecha_texto'])

print("Aplicando magia de Pandas para arreglar las fechas (esto puede tardar unos segundos)...")

# 3. Dejamos que Pandas intente adivinar el formato de fecha (es experto en esto)
df_temp['fecha_real'] = pd.to_datetime(df_temp['fecha_texto'], format='mixed', dayfirst=True, errors='coerce')

# 4. Ahora sí, extraemos el año y mes numérico limpio
df_temp['año'] = df_temp['fecha_real'].dt.year
df_temp['mes_numero'] = df_temp['fecha_real'].dt.month

# 5. Volvemos a agrupar todo por mes (porque el día ya no nos importa)
df_viajes_mensual = df_temp.groupby(['año', 'mes_numero'])['total_viajes'].sum().reset_index()

# Limpiamos los nulos por las dudas y convertimos a entero
df_viajes_mensual = df_viajes_mensual.dropna()
df_viajes_mensual['año'] = df_viajes_mensual['año'].astype(int)
df_viajes_mensual['mes_numero'] = df_viajes_mensual['mes_numero'].astype(int)

# Ordenamos cronológicamente
df_viajes_mensual = df_viajes_mensual.sort_values(['año', 'mes_numero']).reset_index(drop=True)

print("¡Dataset mensual consolidado y limpio!")
display(df_viajes_mensual.head())

Procesando los archivos masivos ignorando problemas de fechas en CSV...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aplicando magia de Pandas para arreglar las fechas (esto puede tardar unos segundos)...
¡Dataset mensual consolidado y limpio!


,año,mes_numero,total_viajes
0,2014,1,17080258.0
1,2014,2,18068193.0
2,2014,3,20689889.0
3,2014,4,21366758.0
4,2014,5,22648413.0
